pipeline 功能理解

In [1]:

# 进阶：将 PCA 用于真实机器学习模型
# PCA 不仅可以画图，也可以作为其他机器学习模型的预处理步骤，例如：
# 原始特征 → 标准化 → PCA 降维 → LogisticRegression 分类

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=23,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

pred_y = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_y))
print("PCA 保留的方差比例：", model.named_steps["pca"].explained_variance_ratio_.sum())

Accuracy: 0.9
PCA 保留的方差比例： 0.9574419222931336


In [ ]:
# 一般来说，Pipeline 的中间步骤需要实现 fit 和 ；最后一个步骤只需要支持 fit，因为它是最终估计器。最后一步也可以是转换器，但本例的最后一步是分类器。

# 可以放到pipeline里面的中间对象（除最后一个对象以前的所有对象）必须要有：训练方法 fit() 和 转化方法 transform()。最后一步可以是分类器、回归器，也可以是其他估计器
# pipeline 里面每个步骤定义的格式必须是：("步骤名称", 对象)
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("classifier", LogisticRegression(max_iter=1000))
])

# 拆分以上步骤, 这就是 Pipeline 的核心工作流程。
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_pca, y_train)


In [5]:
# 创建 Pipeline 后，你可以通过 named_steps 访问每个对象。
# 获取标准化器
scaler = pipeline.named_steps["scaler"]

# 获取 PCA 对象
pca = pipeline.named_steps["pca"]

# 获取逻辑回归分类器
classifier = pipeline.named_steps["classifier"]

# 拿到每一个部分的对象之后就可以取出每个对象的数据（这些属性通常需要先执行 pipeline.fit(X_train, y_train) 才能读取。）

In [7]:
# pipeline.predict 的时候对于StandardScaler和PCA这类的数据处理过程的无监督方法
# 调用 pipeline.predict 的时候只会使用内部的 transform 方法，因为预测阶段不会去改变训练时训练好的 scaler 和 pca 的相关参数了，所以不会调用 fit。只进行数据转化不训练数据

# 训练时：X_train → fit_transform → fit_transform → fit(X, y)
# 预测时： X_test → transform → transform → predict(X)

# 使用 SimpleImputer、OneHotEncoder、TfidfVectorizer 等转换器时，同样适用以上原理


In [17]:
# 查看 pipeline 里面包含哪些步骤
print(pipeline.named_steps.keys())

for key in pipeline.named_steps.keys():
    print("key: ", key)
for step in pipeline.named_steps.values():
    print("step: ", step)
for (key, step) in zip(pipeline.named_steps.keys(), pipeline.named_steps.values()):
    print(f"{key} -> {step}")

# 也可以使用索引访问
print(pipeline[0])   # StandardScaler
print(pipeline[1])   # PCA
print(pipeline[-1])  # LogisticRegression

# 构造新的流水线
print(pipeline[:-1])

dict_keys(['scaler', 'pca', 'classifier'])
key:  scaler
key:  pca
key:  classifier
step:  StandardScaler()
step:  PCA(n_components=2)
step:  LogisticRegression(max_iter=1000)
scaler -> StandardScaler()
pca -> PCA(n_components=2)
classifier -> LogisticRegression(max_iter=1000)
StandardScaler()
PCA(n_components=2)
LogisticRegression(max_iter=1000)
Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=2))])


In [18]:
"""
三、Pipeline 的核心方法：fit、transform、predict
不同对象虽然都遵循 sklearn 的统一接口，但它们支持的方法并不完全相同。
方法	Pipeline            中的作用
fit(X, y)	            按顺序训练整个流程
fit_transform(X, y)	    训练并转换数据；仅当最后一步支持转换时可用
transform(X)	        使用已学习的转换参数处理数据；要求最后一步也支持转换
predict(X)	            经过前面的转换步骤后，调用最终模型预测
predict_proba(X)	    经过前面的转换步骤后，调用最终模型输出类别概率；最终模型需要支持该方法
decision_function(X)	调用最终模型输出决策分数；最终模型需要支持该方法
score(X, y)	            调用最终估计器的评分方法
"""


'\n三、Pipeline 的核心方法：fit、transform、predict\n不同对象虽然都遵循 sklearn 的统一接口，但它们支持的方法并不完全相同。\n方法\tPipeline            中的作用\nfit(X, y)\t            按顺序训练整个流程\nfit_transform(X, y)\t    训练并转换数据；仅当最后一步支持转换时可用\ntransform(X)\t        使用已学习的转换参数处理数据；要求最后一步也支持转换\npredict(X)\t            经过前面的转换步骤后，调用最终模型预测\npredict_proba(X)\t    经过前面的转换步骤后，调用最终模型输出类别概率；最终模型需要支持该方法\ndecision_function(X)\t调用最终模型输出决策分数；最终模型需要支持该方法\nscore(X, y)\t            调用最终估计器的评分方法\n\n\n'

In [19]:
# 查询所有参数
params = model.get_params()
for name, value in params.items():
    print(name, value)

# 使用 set_params() 修改步骤参数

# 禁用 PCA 步骤
model.set_params(pca="passthrough")

# 移除某个步骤
model.set_params(pca=None)

memory None
steps [('scaler', StandardScaler()), ('pca', PCA(n_components=2)), ('classifier', LogisticRegression(max_iter=1000))]
transform_input None
verbose False
scaler StandardScaler()
pca PCA(n_components=2)
classifier LogisticRegression(max_iter=1000)
scaler__copy True
scaler__with_mean True
scaler__with_std True
pca__copy True
pca__iterated_power auto
pca__n_components 2
pca__n_oversamples 10
pca__power_iteration_normalizer auto
pca__random_state None
pca__svd_solver auto
pca__tol 0.0
pca__whiten False
classifier__C 1.0
classifier__class_weight None
classifier__dual False
classifier__fit_intercept True
classifier__intercept_scaling 1
classifier__l1_ratio 0.0
classifier__max_iter 1000
classifier__n_jobs None
classifier__penalty deprecated
classifier__random_state None
classifier__solver lbfgs
classifier__tol 0.0001
classifier__verbose 0
classifier__warm_start False


In [20]:
# Pipeline 最重要的价值：防止数据泄漏

# 即一上来就对权量数据进行标准化
# 不推荐：在切分数据之前学习了全部数据的统计量
# X_scaled = StandardScaler().fit_transform(X)
# 正确的操作是 fit_transform 只对X_train进行操作，而对 X_test 只使用 transform
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y_train)
lr.predict(X_test_scaled)



array([0, 0, 0, 1, 2, 1, 2, 1, 0, 2, 1, 2, 0, 2, 2, 1, 2, 1, 1, 2, 0, 2,
       2, 0, 0, 0, 0, 1, 1, 1])

In [21]:
# Pipeline 与 ColumnTransformer
# 数据里面既有数值特征又有类别特征的时候，想对数值特征进行 StandardScaler 处理，而希望对类别特征进行 OneHotEncoder 处理，因此此时就要使用到 ColumnTransformer 来进行分别处理
# Pipeline 与 ColumnTransformer 这两个工具的关系是：
# - Pipeline：按顺序组织多个处理步骤。
# - ColumnTransformer：对不同列分别执行不同的处理流程。
# - 它们可以嵌套使用。

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# 示例数据
df = pd.DataFrame({
    "age": [25, 35, 42, None, 30, 50],
    "income": [12000, 20000, 18000, 15000, None, 26000],
    "city": ["北京", "上海", "深圳", "北京", None, "上海"],
    "gender": ["男", "女", "男", "女", "男", None]
})

y = [0, 1, 1, 0, 0, 1]

numeric_features = ["age", "income"]
categorical_features = ["city", "gender"]

# 数值特征处理流水线
numeric_pipeline = Pipeline([
    # 对 Nan 数据进行均值填充处理
    ("imputer", SimpleImputer(strategy="mean")),
    # 对数据进行标准化
    ("scaler", StandardScaler())
])

# 类别特征处理流水线
categorical_pipeline = Pipeline([
    # 对 Nan 数据进行出现次数最多的数据填充
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # 对类别数据进行独热编码
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 按列选择不同的流水线，以对不同的数据进行处理
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# 完整的机器学习流水线
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(df, y)

print("预测结果：", model.predict(df))
print("处理后特征矩阵形状：", model.named_steps["preprocessor"].transform(df).shape)




预测结果： [0 1 1 0 0 1]
处理后特征矩阵形状： (6, 7)
